In [48]:
# Loading libraries and files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [49]:
# Define the file path: Please change it
file_path = r"ratings.csv"

# Read the CSV file into a DataFrame
ratings = pd.read_csv(file_path)

# Display the first few rows of the DataFrame to verify it has loaded correctly
print(ratings.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [50]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [51]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(ratings, test_size = 0.30, random_state = 1)

print(X_train.shape)
print(X_test.shape)

(70585, 4)
(30251, 4)


In [52]:
# Pivot ratings into movie features
user_data = X_train.pivot(index = 'movieId', columns = 'userId', values = 'rating').fillna(0)
user_data.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,0.0,0.0,0.0,0.0,4.5,0.0,0.0,0.0,...,4.0,0.0,0.0,3.0,0.0,2.5,0.0,0.0,3.0,5.0
2,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0


### Create a Copy of train and test dataset
These datasets will be used for prediction and evaluation.

Dummy train will be used later for prediction of the movies which has not been rated by the user. To ignore the movies rated by the user, we will mark it as 0 during prediction. The movies not rated by user are marked as 1 for prediction.

Dummy test will be used for evaluation. To evaluate, we will only make prediction on the movies rated by the user. So, this is marked as 1. This is the opposite of dummy_train

In [53]:
# Make a copy of train and test datasets
dummy_train = X_train.copy()
dummy_test = X_test.copy()

dummy_train['rating'] = dummy_train['rating'].apply(lambda x: 0 if x > 0 else 1)
dummy_test['rating'] = dummy_test['rating'].apply(lambda x: 1 if x > 0 else 0)

In [54]:
# The movies not rated by user are marked as 1 for prediction
dummy_train = dummy_train.pivot(index = 'movieId', columns = 'userId', values = 'rating').fillna(1)

# The movies not rated by user are marked as 0 for evaluation
dummy_test = dummy_test.pivot(index ='movieId', columns = 'userId', values = 'rating').fillna(0)

## User-User Similarity matrix Using Cosine Similarity

In [55]:
# Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity

# User Similarity Matrix which uses Cosine similarity as a similarity measure between Users
user_similarity = cosine_similarity(user_data)
user_similarity[np.isnan(user_similarity)] = 0
print(user_similarity)
print(user_similarity.shape)

[[1.         0.23372693 0.20338434 ... 0.         0.         0.        ]
 [0.23372693 1.         0.24698794 ... 0.         0.         0.        ]
 [0.20338434 0.24698794 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         1.         0.        ]
 [0.         0.         0.         ... 1.         1.         0.        ]
 [0.         0.         0.         ... 0.         0.         1.        ]]
(8531, 8531)


In [56]:
# Predicting the User ratings on the movies
user_predicted_ratings = np.dot(user_similarity, user_data)
user_predicted_ratings

array([[1.49754387e+02, 1.31288543e+01, 5.86813792e+00, ...,
        3.37861284e+02, 1.93294994e+01, 5.53207979e+02],
       [1.27814198e+02, 1.24360135e+01, 5.88159412e+00, ...,
        3.61905597e+02, 1.65631192e+01, 4.06445671e+02],
       [9.11879284e+01, 5.03289208e+00, 4.74340930e+00, ...,
        2.51944418e+02, 1.12062696e+01, 1.90234210e+02],
       ...,
       [0.00000000e+00, 1.99707443e+00, 0.00000000e+00, ...,
        1.22794555e+00, 0.00000000e+00, 1.89764655e+01],
       [0.00000000e+00, 1.99707443e+00, 0.00000000e+00, ...,
        1.22794555e+00, 0.00000000e+00, 1.89764655e+01],
       [1.41114849e+00, 3.01131609e+00, 0.00000000e+00, ...,
        2.08469288e+01, 3.36761267e-01, 3.87654941e+01]],
      shape=(8531, 610))

In [57]:
user_predicted_ratings.shape

(8531, 610)

In [58]:
# np.multiply for cell-by-cell multiplication
user_final_ratings = np.multiply(user_predicted_ratings, dummy_train)
user_final_ratings.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.000000,13.128854,5.868138,87.672611,22.888569,114.739011,0.000000,36.683375,14.847323,43.146697,...,0.000000,63.922791,262.993024,0.000000,82.867143,0.000000,100.849144,337.861284,0.000000,0.000000
2,127.814198,12.436013,5.881594,76.042237,23.487144,0.000000,76.345506,37.135939,13.715646,40.511768,...,44.774927,65.490391,234.163904,0.000000,0.000000,306.286602,95.251225,0.000000,16.563119,406.445671
3,91.187928,5.032892,4.743409,47.336363,17.057341,0.000000,40.148693,26.468031,8.231939,16.970573,...,16.792466,52.339832,155.864949,37.923197,41.349821,182.180659,70.344621,0.000000,11.206270,190.234210
4,19.604879,0.974115,0.958548,20.604536,11.866120,0.000000,6.794177,11.643058,4.023811,2.972971,...,4.193122,27.050598,62.520468,21.282445,11.920082,59.584358,19.402669,43.509307,4.438226,30.583355
5,68.968786,7.034153,3.159544,44.738622,18.129831,0.000000,37.901937,24.029267,7.219736,24.300391,...,22.377478,49.560138,137.653233,0.000000,44.558656,176.949454,52.333260,178.334642,10.241423,200.282065


In [59]:
# Top 5 movie recommendations for the User 29
user_final_ratings.iloc[29].sort_values(ascending = False)[0:5]

userId
414    187.126159
474    178.485060
599    128.567999
606    115.002980
275    105.214351
Name: 30, dtype: float64

## Evaluation

Evaluation will be the same as you have seen above for the prediction. The only difference being, you will evaluate for the movie already rated by the User instead of predicting it for the movie not rated by the user.

In [60]:
test_user_features = X_test.pivot(index = 'movieId', columns = 'userId', values = 'rating').fillna(0)
test_user_similarity = cosine_similarity(test_user_features)
test_user_similarity[np.isnan(test_user_similarity)] = 0

print(test_user_similarity)
print("- "*10)
print(test_user_similarity.shape)

[[1.         0.05868362 0.02169197 ... 0.         0.         0.        ]
 [0.05868362 1.         0.03474688 ... 0.         0.         0.        ]
 [0.02169197 0.03474688 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         1.         0.        ]
 [0.         0.         0.         ... 1.         1.         0.        ]
 [0.         0.         0.         ... 0.         0.         1.        ]]
- - - - - - - - - - 
(6123, 6123)


In [61]:
user_predicted_ratings_test = np.dot(test_user_similarity, test_user_features)
user_predicted_ratings_test

array([[2.87830944e+01, 2.16248801e+00, 9.03976171e-01, ...,
        7.45974616e+01, 4.05287413e+00, 4.90485850e+01],
       [2.29182523e+01, 2.28919669e+00, 3.47817232e-01, ...,
        4.25987039e+01, 2.47788787e+00, 6.29212415e+01],
       [6.48471839e+01, 1.22517305e+00, 5.33987126e-02, ...,
        3.66692283e+01, 1.69683821e+00, 4.22367860e+01],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        5.70260225e-01, 4.56208180e-01, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        5.70260225e-01, 4.56208180e-01, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 4.07222247e+00]],
      shape=(6123, 610))

In [62]:
test_user_final_rating = np.multiply(user_predicted_ratings_test, dummy_test)
test_user_final_rating.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.000000,0.0,0.0,0.0,11.783358,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,108.558628,0.0,31.986677,0.0,28.960342,74.597462,0.0,0.0
2,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,8.202154,0.0,0.0,...,0.0,23.173124,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0
3,64.847184,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0
4,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0
5,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0


In [63]:
ratings['rating'].describe()

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

In [64]:
from sklearn.preprocessing import MinMaxScaler

X = test_user_final_rating.copy()
X = X[X > 0] # Only consider non-zero values as 0 means the user has not rated the movies

scaler = MinMaxScaler(feature_range = (0.5, 5))
scaler.fit(X)
pred = scaler.transform(X)

print(pred)

[[       nan        nan        nan ... 1.54183291        nan        nan]
 [       nan        nan        nan ...        nan        nan        nan]
 [1.91105701        nan        nan ...        nan        nan        nan]
 ...
 [       nan        nan        nan ...        nan        nan        nan]
 [       nan        nan        nan ...        nan        nan        nan]
 [       nan        nan        nan ...        nan        nan        nan]]


In [65]:
# Total of non-NaN values
total_non_nan = np.count_nonzero(~np.isnan(pred))
total_non_nan

np.int64(30251)

In [66]:
test = X_test.pivot(index = 'movieId', columns = 'userId', values = 'rating')
test.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,4.0,NaN,4.0,2.5,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,...,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
# RMSE Score User-User Based Collaborative Filtering

diff_sqr_matrix = (test - pred)**2
sum_of_squares_err = diff_sqr_matrix.sum().sum() # df.sum().sum() by default ignores null values

rmse = np.sqrt(sum_of_squares_err/total_non_nan)
print(rmse)

1.7695421938800078
